In [ ]:
import pandas as pd

In [ ]:
#Load data
all_experience = pd.read_pickle("/shared/share_scp/coresignal/all_experience_AnalysisFile_latest.pkl")

In [ ]:
owner_founder_columns = [col for col in all_experience.columns if 'own' in col or 'found' in col]

mask = all_experience[owner_founder_columns].any(axis=1)
startup_events = all_experience[mask].copy()


In [ ]:
all_experience.columns

In [ ]:
scp_data = pd.read_csv("/shared/share_scp/coresignal/gitrepo_facebook2/SCP_dataset_minimal.csv")


In [ ]:
import re

# Words to ignore: articles/connectors and common legal suffixes only
DROP_TOKENS = {
    "a", "an", "the",
    "and", "&", "of", "for", "to",
    "co", "company",
    "inc", "incorporated", "corp", "corporation",
    "ltd", "limited", "llc", "plc", "lp", "llp",
    "gmbh", "ag", "sa", "sas", "sarl", "bv", "nv",
    "pte", "pty", "oy", "ab", "as",
}

def normalize_company_name(name):
    """Normalize company names using only allowed flexibility rules."""
    if pd.isna(name):
        return pd.NA

    s = str(name).lower().strip()
    # Keep only alphanumeric content; treat punctuation as separators.
    s = re.sub(r"[^a-z0-9]+", " ", s)
    tokens = [tok for tok in s.split() if tok and tok not in DROP_TOKENS]

    # Exact matching is done on this canonical form (no fuzzy logic).
    if not tokens:
        return pd.NA
    return " ".join(tokens)

# Build normalized keys
all_experience["company_name_norm"] = all_experience["company_name"].map(normalize_company_name)
scp_data["entity_name_norm"] = scp_data["entity_name"].map(normalize_company_name)

# Optional deduped lookup of SCP entities by normalized name
scp_lookup = (
    scp_data.loc[scp_data["entity_name_norm"].notna(), ["entity_name", "entity_name_norm"]]
    .drop_duplicates(subset=["entity_name_norm"])
    .rename(columns={"entity_name": "entity_name_matched"})
)

# Match all_experience to SCP using normalized exact key
all_experience_matched = all_experience.merge(
    scp_lookup,
    left_on="company_name_norm",
    right_on="entity_name_norm",
    how="left",
    validate="m:1",
)

# Quick check: how many rows got a match
match_rate = all_experience_matched["entity_name_matched"].notna().mean()
print(f"Match rate: {match_rate:.2%}")

In [ ]:
# Comprehensive match diagnostics: statistics + examples for all major match types
import numpy as np

# 1) SCP-side ambiguity by normalized key (before deduplication)
scp_norm_summary = (
    scp_data.loc[scp_data["entity_name_norm"].notna(), ["entity_name_norm", "entity_name"]]
    .drop_duplicates()
    .groupby("entity_name_norm", as_index=False)
    .agg(
        n_scp_names=("entity_name", "nunique"),
        scp_name_examples=("entity_name", lambda s: list(s.head(10)))
    )
)

ambiguous_scp_norm = scp_norm_summary[scp_norm_summary["n_scp_names"] > 1].copy()
ambiguous_keys = set(ambiguous_scp_norm["entity_name_norm"])

# 2) all_experience-side match categorization
ae = all_experience_matched.copy()

ae["matched"] = ae["entity_name_matched"].notna()
ae["company_name_str"] = ae["company_name"].astype("string")
ae["entity_name_matched_str"] = ae["entity_name_matched"].astype("string")

# Case/space-only exactness check (no normalization token drops)
ae["exact_ignore_case_space"] = (
    ae["company_name_str"].str.strip().str.casefold()
    == ae["entity_name_matched_str"].str.strip().str.casefold()
)

# Empty/invalid normalized keys
ae["empty_norm"] = ae["company_name_norm"].isna()

# Did this row's normalized key have multiple SCP candidates?
ae["ambiguous_norm_key"] = ae["company_name_norm"].isin(ambiguous_keys)

# Final mutually-exclusive category label
ae["match_type"] = np.select(
    [
        ae["matched"] & ae["exact_ignore_case_space"],
        ae["matched"] & ~ae["exact_ignore_case_space"],
        ~ae["matched"] & ae["empty_norm"],
        ~ae["matched"] & ~ae["empty_norm"],
    ],
    [
        "matched_exact_ignore_case_space",
        "matched_via_normalization_rules",
        "unmatched_empty_normalized_name",
        "unmatched_nonempty_normalized_name",
    ],
    default="other",
)

# 3) Summary statistics
total_rows = len(ae)
summary_counts = ae["match_type"].value_counts(dropna=False).rename_axis("match_type").to_frame("rows")
summary_counts["share"] = summary_counts["rows"] / total_rows

print("=" * 80)
print("OVERALL MATCH SUMMARY")
print("=" * 80)
print(f"Total all_experience rows: {total_rows:,}")
print(f"Matched rows: {ae['matched'].sum():,} ({ae['matched'].mean():.2%})")
print()
print(summary_counts)

print("\n" + "=" * 80)
print("SCP NORMALIZED-KEY AMBIGUITY")
print("=" * 80)
print(f"Distinct normalized SCP keys: {len(scp_norm_summary):,}")
print(f"Ambiguous normalized SCP keys (>1 SCP name): {len(ambiguous_scp_norm):,}")
print(f"Rows in all_experience using ambiguous keys: {ae['ambiguous_norm_key'].sum():,}")
if ae["matched"].any():
    ambiguous_matched = ae.loc[ae["matched"], "ambiguous_norm_key"].mean()
    print(f"Share of matched rows with ambiguous key: {ambiguous_matched:.2%}")

# 4) Examples for each match type
print("\n" + "=" * 80)
print("EXAMPLES BY MATCH TYPE")
print("=" * 80)
example_cols = ["company_name", "company_name_norm", "entity_name_matched", "entity_name_norm", "ambiguous_norm_key"]
for t in [
    "matched_exact_ignore_case_space",
    "matched_via_normalization_rules",
    "unmatched_nonempty_normalized_name",
    "unmatched_empty_normalized_name",
]:
    sub = ae.loc[ae["match_type"] == t, example_cols].head(10)
    print(f"\n{t} -> {len(ae[ae['match_type'] == t]):,} rows")
    if len(sub) == 0:
        print("  (no rows)")
    else:
        print(sub.to_string(index=False))

# 5) Examples of ambiguous SCP mappings with all_experience impact
print("\n" + "=" * 80)
print("AMBIGUOUS KEY EXAMPLES (SCP side)")
print("=" * 80)

if len(ambiguous_scp_norm) == 0:
    print("No ambiguous normalized keys found in scp_data.")
else:
    # Add count of all_experience rows hitting each ambiguous key
    ae_amb_counts = (
        ae.loc[ae["ambiguous_norm_key"] & ae["company_name_norm"].notna(), "company_name_norm"]
        .value_counts()
        .rename_axis("entity_name_norm")
        .to_frame("all_experience_rows")
        .reset_index()
)

    ambiguous_report = (
        ambiguous_scp_norm.merge(ae_amb_counts, on="entity_name_norm", how="left")
        .fillna({"all_experience_rows": 0})
        .sort_values(["all_experience_rows", "n_scp_names"], ascending=[False, False])
)

    print(ambiguous_report.head(20).to_string(index=False))

    # Detailed row-level examples from all_experience for top ambiguous keys
    top_amb_keys = ambiguous_report.head(5)["entity_name_norm"].tolist()
    for k in top_amb_keys:
        print("\n" + "-" * 80)
        print(f"Ambiguous normalized key: {k}")
        candidates = ambiguous_scp_norm.loc[ambiguous_scp_norm["entity_name_norm"] == k, "scp_name_examples"].iloc[0]
        print(f"SCP candidate names: {candidates}")
        ex = ae.loc[ae["company_name_norm"] == k, ["company_name", "company_name_norm", "entity_name_matched"]].head(10)
        print("all_experience examples:")
        print(ex.to_string(index=False))

# 6) Optional reusable outputs
match_summary_table = summary_counts.reset_index()
match_examples = ae[example_cols + ["match_type"]].copy()
ambiguous_key_table = ambiguous_scp_norm.copy()